# Noise Banding Quantification

Explore row-wise diagnostics for frames with horizontal black/white banding. The example frame is `img_20260423_030001.png`. The goal is to find quantities that flag corrupted horizontal rows so those pixels can later be masked or set to `NaN` before ROI statistics are computed.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["image.cmap"] = "gray"

data_path = Path("/Volumes/general/garage")
bad_path = data_path / "img_20260423_030001.png"
comparison_paths = [
    bad_path,
    data_path / "img_20260423_030501.png",
    data_path / "img_20260423_152501.png",
    data_path / "img_20260423_091501.png",
    data_path / "img_20260422_215501.png"
]
[p for p in comparison_paths if p.exists()]

In [ ]:
def load_rgb(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB"), dtype=np.float32)


def to_luminance(rgb):
    return 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]


def downsample_for_display(image, factor=8):
    return image[::factor, ::factor]


def row_metrics(path, local_window=101):
    rgb = load_rgb(path)
    lum = to_luminance(rgb)

    row_rgb_median = np.median(rgb, axis=1)
    row_lum_median = np.median(lum, axis=1)
    row_lum_p10 = np.percentile(lum, 10, axis=1)
    row_lum_p90 = np.percentile(lum, 90, axis=1)
    row_lum_iqr = np.percentile(lum, 75, axis=1) - np.percentile(lum, 25, axis=1)
    row_black_fraction = (lum < 8).mean(axis=1)
    row_white_fraction = (lum > 220).mean(axis=1)
    row_delta = np.r_[0, np.abs(np.diff(row_lum_median))]
    local_baseline = pd.Series(row_lum_median).rolling(local_window, center=True, min_periods=1).median().to_numpy()
    local_residual = row_lum_median - local_baseline

    df = pd.DataFrame(
        {
            "y": np.arange(lum.shape[0]),
            "r_median": row_rgb_median[:, 0],
            "g_median": row_rgb_median[:, 1],
            "b_median": row_rgb_median[:, 2],
            "rgb_median_spread": row_rgb_median.max(axis=1) - row_rgb_median.min(axis=1),
            "lum_median": row_lum_median,
            "lum_p10": row_lum_p10,
            "lum_p90": row_lum_p90,
            "lum_iqr": row_lum_iqr,
            "black_fraction": row_black_fraction,
            "white_fraction": row_white_fraction,
            "row_delta": row_delta,
            "local_baseline": local_baseline,
            "local_residual": local_residual,
            "abs_local_residual": np.abs(local_residual),
        }
    )
    return rgb, lum, df


def rolling_count(mask, window):
    return np.convolve(np.asarray(mask, dtype=float), np.ones(window), mode="same")


def dilate_row_mask(mask, radius=2):
    mask = np.asarray(mask, dtype=bool)
    out = mask.copy()
    for shift in range(1, radius + 1):
        out[:-shift] |= mask[shift:]
        out[shift:] |= mask[:-shift]
    return out


def fill_small_gaps(mask, max_gap=12):
    out = np.asarray(mask, dtype=bool).copy()
    i = 0
    while i < len(out):
        if out[i]:
            i += 1
            continue
        start = i
        while i < len(out) and not out[i]:
            i += 1
        if start > 0 and i < len(out) and i - start <= max_gap:
            out[start:i] = True
    return out


def remove_small_runs(mask, min_len=3):
    out = np.asarray(mask, dtype=bool).copy()
    i = 0
    while i < len(out):
        if not out[i]:
            i += 1
            continue
        start = i
        while i < len(out) and out[i]:
            i += 1
        if i - start < min_len:
            out[start:i] = False
    return out


def band_seed_mask_from_row_metrics(df, local_residual_threshold=30, row_delta_threshold=20):
    local_anomaly = df["abs_local_residual"] > local_residual_threshold
    edge_anomaly = df["row_delta"] > row_delta_threshold
    flat_black_or_white = (df["lum_iqr"] < 8) & ((df["black_fraction"] > 0.5) | (df["white_fraction"] > 0.5))
    return (local_anomaly | edge_anomaly | flat_black_or_white).to_numpy()


def band_tophat_masks_from_row_metrics(
    df,
    row_delta_threshold=20,
    evidence_window=31,
    evidence_min_count=2,
    dilation_radius=4,
    fill_gap=50,
    min_run_len=8,
):
    """Create a broad horizontal-band mask from sparse edge seeds.

    The raw row-delta threshold catches band edges, but can miss the middle of a
    corrupted band. This builds a top-hat-like region by requiring a local
    cluster of edge/flat-row seeds, dilating it, filling short gaps, and removing
    tiny isolated runs. Daytime comparison frames tested here have row_delta < 20
    and produce no mask.
    """
    edge_seed = (df["row_delta"] > row_delta_threshold).to_numpy()
    flat_seed = ((df["lum_iqr"] < 8) & ((df["black_fraction"] > 0.5) | (df["white_fraction"] > 0.5))).to_numpy()
    seeds = edge_seed | flat_seed
    evidence = rolling_count(seeds, evidence_window) >= evidence_min_count
    band = dilate_row_mask(evidence, radius=dilation_radius)
    band = fill_small_gaps(band, max_gap=fill_gap)
    band = remove_small_runs(band, min_len=min_run_len)
    return {
        "edge_seed": edge_seed,
        "flat_seed": flat_seed,
        "seeds": seeds,
        "evidence": evidence,
        "band": band,
    }


def band_mask_from_row_metrics(df):
    return band_tophat_masks_from_row_metrics(df)["band"]


def true_runs(mask, min_len=1):
    runs = []
    start = None
    for i, value in enumerate(mask):
        if value and start is None:
            start = i
        elif not value and start is not None:
            if i - start >= min_len:
                runs.append((start, i - 1, i - start))
            start = None
    if start is not None and len(mask) - start >= min_len:
        runs.append((start, len(mask) - 1, len(mask) - start))
    return runs

## Visual Check

The bad frame is compared with the next available frame and a known normal frame. The downsampled views are only for visual orientation; row metrics below are computed on the full-resolution image.

In [ ]:
loaded = []
for path in comparison_paths:
    if path.exists():
        rgb, lum, df = row_metrics(path)
        masks = band_tophat_masks_from_row_metrics(df)
        seed_mask = band_seed_mask_from_row_metrics(df)
        loaded.append({"path": path, "rgb": rgb, "lum": lum, "rows": df, "mask": masks["band"], "seed_mask": seed_mask, "masks": masks})

fig, axes = plt.subplots(1, len(loaded), figsize=(5 * len(loaded), 4), squeeze=False)
for ax, item in zip(axes.ravel(), loaded):
    ax.imshow(downsample_for_display(item["rgb"].astype(np.uint8), factor=12))
    ax.set_title(item["path"].name)
    ax.set_axis_off()
plt.tight_layout()

## Summary Metrics

Promising global indicators for banding are:

- high `row_delta` between adjacent row medians,
- high row median residual relative to a rolling vertical median baseline,
- rows that are nearly all black or white across the full width,
- unusual RGB median spread by row.

In [ ]:
summary_rows = []
for item in loaded:
    df = item["rows"]
    mask = item["mask"]
    masks = item["masks"]
    summary_rows.append(
        {
            "filename": item["path"].name,
            "rows": len(df),
            "edge_or_flat_seed_rows": int(masks["seeds"].sum()),
            "evidence_rows": int(masks["evidence"].sum()),
            "band_rows": int(mask.sum()),
            "band_fraction": float(mask.mean()),
            "lum_median_min": df["lum_median"].min(),
            "lum_median_max": df["lum_median"].max(),
            "row_delta_p99": df["row_delta"].quantile(0.99),
            "row_delta_max": df["row_delta"].max(),
            "abs_local_residual_p99": df["abs_local_residual"].quantile(0.99),
            "abs_local_residual_max": df["abs_local_residual"].max(),
            "black_rows_gt_80pct": int((df["black_fraction"] > 0.8).sum()),
            "white_rows_gt_80pct": int((df["white_fraction"] > 0.8).sum()),
            "rgb_spread_p99": df["rgb_median_spread"].quantile(0.99),
            "rgb_spread_max": df["rgb_median_spread"].max(),
        }
    )
summary = pd.DataFrame(summary_rows)
summary

## Row Profiles

The bad frame has large row-to-row jumps and large deviations from a rolling vertical baseline. The normal frames stay smooth by comparison.

In [ ]:
fig, axes = plt.subplots(len(loaded), 4, figsize=(18, 3.2 * len(loaded)), sharex="row")
if len(loaded) == 1:
    axes = axes[None, :]
for row_axes, item in zip(axes, loaded):
    df = item["rows"]
    name = item["path"].name

    row_axes[0].plot(df["y"], df["lum_median"], lw=0.8, label="row median")
    row_axes[0].plot(df["y"], df["local_baseline"], lw=0.8, label="rolling median")
    row_axes[0].set_title(name + chr(10) + "luminance median")
    row_axes[0].legend(fontsize=8)

    row_axes[1].plot(df["y"], df["row_delta"], lw=0.8)
    row_axes[1].axhline(20, color="tab:red", lw=1)
    row_axes[1].set_title("adjacent row median change")

    row_axes[2].plot(df["y"], df["abs_local_residual"], lw=0.8)
    row_axes[2].axhline(30, color="tab:red", lw=1)
    row_axes[2].set_title("abs local residual")

    row_axes[3].plot(df["y"], df["r_median"], color="tab:red", lw=0.7, label="R")
    row_axes[3].plot(df["y"], df["g_median"], color="tab:green", lw=0.7, label="G")
    row_axes[3].plot(df["y"], df["b_median"], color="tab:blue", lw=0.7, label="B")
    row_axes[3].set_title("RGB row medians")
    row_axes[3].legend(fontsize=8)

    for ax in row_axes:
        ax.grid(True, alpha=0.25)
        ax.set_xlabel("row y")
plt.tight_layout()

## Candidate Row Mask

The mask below is now edge-seeded and region-based. A raw `row_delta > 20` seed catches the abrupt edges of noisy bands. A rolling count creates top-hat-like support around regions with multiple nearby edge seeds, then dilation and gap filling cover the whole band even where row deltas are low in the middle.

In [ ]:
bad_item = loaded[0]
bad_df = bad_item["rows"]
simple_seed_mask = bad_item["seed_mask"]
masks = bad_item["masks"]
row_mask = masks["band"]

runs = true_runs(row_mask, min_len=2)
print(f"Simple seed rows, including local residual: {simple_seed_mask.sum()} / {len(simple_seed_mask)}")
print(f"Edge/flat seed rows used for top-hat: {masks['seeds'].sum()} / {len(row_mask)}")
print(f"Top-hat evidence rows: {masks['evidence'].sum()} / {len(row_mask)}")
print(f"Final band rows: {row_mask.sum()} / {len(row_mask)}")
print(f"Number of final row runs: {len(runs)}")
pd.DataFrame(runs, columns=["start_y", "end_y", "n_rows"]).head(30)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
rgb_small = downsample_for_display(bad_item["rgb"].astype(np.uint8), factor=8)
axes[0].imshow(rgb_small)
axes[0].set_title("bad frame, downsampled")
axes[0].set_axis_off()

mask_small = row_mask[::8]
seed_small = simple_seed_mask[::8]
evidence_small = masks["evidence"][::8]

masked_lum = downsample_for_display(bad_item["lum"], factor=8).copy()
masked_lum[mask_small, :] = np.nan
axes[1].imshow(masked_lum, cmap="gray")
axes[1].set_title("luminance with final band rows as NaN")
axes[1].set_axis_off()

axes[2].imshow(rgb_small)
for y in np.flatnonzero(seed_small):
    axes[2].axhline(y, color="yellow", lw=0.35, alpha=0.6)
for y in np.flatnonzero(evidence_small):
    axes[2].axhline(y, color="cyan", lw=0.35, alpha=0.5)
axes[2].set_title("seed rows yellow, evidence cyan")
axes[2].set_axis_off()

axes[3].imshow(rgb_small)
for y in np.flatnonzero(mask_small):
    axes[3].axhline(y, color="red", lw=0.5, alpha=0.7)
axes[3].set_title("final top-hat band mask")
axes[3].set_axis_off()
plt.tight_layout()

## Initial Takeaway

The edge-seeded top-hat mask looks more robust than a raw row-delta threshold. It flags broad noisy-band regions in `img_20260423_030001.png`, while the nighttime and daytime comparison frames tested here produce zero final band rows.

This is the version I would try next in the production ROI path: compute a row mask on the binned or original luminance image, expand it into a 2D mask by row, and ignore masked pixels when computing ROI medians/percentiles.

## Binned Image Test

The production ROI path already works in binned-image coordinates, so it is preferable to detect noisy horizontal bands on the cached `*_bin4.png` images if the signal survives binning. The binned files are grayscale `1152 x 648`, so row runs should be approximately full-resolution row runs divided by four.

In [ ]:
def binned_path_for(path, bin_factor=4):
    return path.parent / "binned" / f"{path.stem}_bin{bin_factor}.png"


def row_metrics_luma(path, local_window=25):
    with Image.open(path) as im:
        lum = np.asarray(im.convert("L"), dtype=np.float32)
    row_lum_median = np.median(lum, axis=1)
    local_baseline = pd.Series(row_lum_median).rolling(local_window, center=True, min_periods=1).median().to_numpy()
    return pd.DataFrame(
        {
            "y": np.arange(lum.shape[0]),
            "lum_median": row_lum_median,
            "lum_iqr": np.percentile(lum, 75, axis=1) - np.percentile(lum, 25, axis=1),
            "black_fraction": (lum < 8).mean(axis=1),
            "white_fraction": (lum > 220).mean(axis=1),
            "row_delta": np.r_[0, np.abs(np.diff(row_lum_median))],
            "local_baseline": local_baseline,
            "abs_local_residual": np.abs(row_lum_median - local_baseline),
        }
    ), lum


def band_tophat_mask_binned_fixed(df):
    masks = band_tophat_masks_from_row_metrics(
        df,
        row_delta_threshold=20,
        evidence_window=9,
        evidence_min_count=1,
        dilation_radius=1,
        fill_gap=13,
        min_run_len=2,
    )
    return masks


def band_tophat_mask_binned_robust(df, abs_floor=25, robust_z=8):
    row_delta = df["row_delta"].to_numpy()
    median = np.median(row_delta)
    mad = np.median(np.abs(row_delta - median))
    scale = 1.4826 * mad if mad > 0 else max(np.std(row_delta), 1.0)
    robust_edge_seed = (row_delta > abs_floor) & ((row_delta - median) / (scale + 1e-6) > robust_z)
    flat_seed = ((df["lum_iqr"] < 8) & ((df["black_fraction"] > 0.5) | (df["white_fraction"] > 0.5))).to_numpy()
    seeds = robust_edge_seed | flat_seed
    evidence = rolling_count(seeds, 9) >= 1
    band = dilate_row_mask(evidence, radius=1)
    band = fill_small_gaps(band, max_gap=13)
    band = remove_small_runs(band, min_len=2)
    return {
        "edge_seed": robust_edge_seed,
        "flat_seed": flat_seed,
        "seeds": seeds,
        "evidence": evidence,
        "band": band,
        "row_delta_median": median,
        "row_delta_mad": mad,
        "row_delta_scale": scale,
    }


binned_rows = []
binned_loaded = []
for path in comparison_paths:
    bp = binned_path_for(path)
    if not bp.exists():
        continue
    df, lum = row_metrics_luma(bp)
    fixed = band_tophat_mask_binned_fixed(df)
    robust = band_tophat_mask_binned_robust(df, abs_floor=25, robust_z=8)
    robust_band_fraction = float(robust["band"].mean())
    robust_gated_band = robust["band"] if robust_band_fraction >= 0.05 else np.zeros_like(robust["band"], dtype=bool)
    binned_loaded.append({"path": bp, "rows": df, "lum": lum, "fixed": fixed, "robust": robust, "robust_gated_band": robust_gated_band})
    binned_rows.append(
        {
            "filename": bp.name,
            "rows": len(df),
            "row_delta_max": df["row_delta"].max(),
            "row_delta_p99": df["row_delta"].quantile(0.99),
            "abs_local_residual_max": df["abs_local_residual"].max(),
            "fixed_band_rows": int(fixed["band"].sum()),
            "robust_band_rows": int(robust["band"].sum()),
            "robust_band_fraction": robust_band_fraction,
            "robust_gated_band_rows": int(robust_gated_band.sum()),
            "robust_seed_rows": int(robust["seeds"].sum()),
        }
    )

pd.DataFrame(binned_rows)

In [ ]:
bad_binned = binned_loaded[0]
fixed_runs = true_runs(bad_binned["fixed"]["band"], min_len=2)
robust_runs = true_runs(bad_binned["robust"]["band"], min_len=2)
robust_gated_runs = true_runs(bad_binned["robust_gated_band"], min_len=2)
print("Fixed-threshold binned runs")
display(pd.DataFrame(fixed_runs, columns=["start_y_bin", "end_y_bin", "n_rows_bin"]))
print("Robust-threshold binned runs")
display(pd.DataFrame(robust_runs, columns=["start_y_bin", "end_y_bin", "n_rows_bin"]))
print("Robust gated binned runs")
display(pd.DataFrame(robust_gated_runs, columns=["start_y_bin", "end_y_bin", "n_rows_bin"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(bad_binned["lum"], cmap="gray")
axes[0].set_title("bad binned luminance")
axes[0].set_axis_off()

axes[1].imshow(bad_binned["lum"], cmap="gray")
for y in np.flatnonzero(bad_binned["fixed"]["band"]):
    axes[1].axhline(y, color="red", lw=0.5, alpha=0.7)
axes[1].set_title("fixed-threshold band mask")
axes[1].set_axis_off()

axes[2].imshow(bad_binned["lum"], cmap="gray")
for y in np.flatnonzero(bad_binned["robust"]["band"]):
    axes[2].axhline(y, color="red", lw=0.5, alpha=0.7)
axes[2].set_title("robust binned band mask")
axes[2].set_axis_off()
plt.tight_layout()

### Binned Takeaway

The row-band detector works on the cached binned images. The broad bad-frame bands are recovered at approximately one quarter of the full-resolution row coordinates. A fixed binned `row_delta > 20` catches the bad frame well, but can fire on strong real daytime edges. The robust binned variant adds a per-frame robust z-score and an absolute floor; adding a simple frame-level gate such as `robust_band_fraction >= 0.05` keeps the bad frame while suppressing the small daytime false positive in this comparison set.

This is the version I would use in the binned ROI extraction path: run on the binned luminance image, apply the robust top-hat mask only if it covers a meaningful fraction of rows, then set those binned rows to `NaN` before ROI statistics.